# Assignment: Representation Learning using Variants of Autoencoders

This notebook covers:
- **Task 1**: Standard PCA vs Randomized PCA with 30 principal components, logistic regression classification, ROC curves, and SNR analysis
- **Task 2**: Single-layer linear autoencoder with tied weights, eigenvector comparison with PCA, and logistic regression
- **Task 3**: Deep convolutional autoencoder, single-layer autoencoder (sigmoid/linear), 3-hidden-layer autoencoder, SNR comparison

## 0. Setup and Data Preparation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
np.random.seed(42)

from sklearn.decomposition import PCA, RandomizedPCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split
from skimage.color import rgb2gray
from skimage.transform import resize

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.callbacks import EarlyStopping

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Load CIFAR-10 and MNIST
(cifar_train_x, cifar_train_y), (cifar_test_x, cifar_test_y) = keras.datasets.cifar10.load_data()
(mnist_train_x, mnist_train_y), (mnist_test_x, mnist_test_y) = keras.datasets.mnist.load_data()

# Merge train/test for our own split
cifar_x = np.concatenate([cifar_train_x, cifar_test_x], axis=0)
cifar_y = np.concatenate([cifar_train_y, cifar_test_y], axis=0).flatten()
mnist_x = np.concatenate([mnist_train_x, mnist_test_x], axis=0)
mnist_y = np.concatenate([mnist_train_y, mnist_test_y], axis=0).flatten()

print(f"CIFAR-10 full: {cifar_x.shape}, labels: {cifar_y.shape}")
print(f"MNIST full: {mnist_x.shape}, labels: {mnist_y.shape}")

In [ ]:
def preprocess_cifar10(images):
    """Convert CIFAR-10 RGB to grayscale and resize to 28x28"""
    processed = []
    for img in images:
        gray = rgb2gray(img)  # Convert to grayscale (0-1 range)
        gray_resized = resize(gray, (28, 28), anti_aliasing=True)
        processed.append(gray_resized)
    return np.array(processed)

def preprocess_mnist(images):
    """MNIST is already 28x28 grayscale, just ensure float"""
    return images.astype('float32') / 255.0

def normalize_intensity(images, low=50, high=200):
    """Normalize intensity between low and high"""
    img_min = images.min()
    img_max = images.max()
    normalized = (images - img_min) / (img_max - img_min)  # 0 to 1
    normalized = normalized * (high - low) + low  # Scale to [low, high]
    return normalized

print("Preprocessing CIFAR-10...")
cifar_x_proc = preprocess_cifar10(cifar_x)
print(f"CIFAR-10 after grayscale+resize: {cifar_x_proc.shape}, range: [{cifar_x_proc.min():.3f}, {cifar_x_proc.max():.3f}]")

print("Preprocessing MNIST...")
mnist_x_proc = preprocess_mnist(mnist_x)
print(f"MNIST after preprocessing: {mnist_x_proc.shape}, range: [{mnist_x_proc.min():.3f}, {mnist_x_proc.max():.3f}]")

# Normalize intensities to [50, 200]
cifar_x_norm = normalize_intensity(cifar_x_proc, 50, 200)
mnist_x_norm = normalize_intensity(mnist_x_proc, 50, 200)

print(f"\nCIFAR-10 normalized: range [{cifar_x_norm.min():.2f}, {cifar_x_norm.max():.2f}]")
print(f"MNIST normalized: range [{mnist_x_norm.min():.2f}, {mnist_x_norm.max():.2f}]")

In [ ]:
# Split: 70% train, 20% validation, 10% test
def split_data(X, y):
    # First split: 70% train, 30% temp
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    # Second split: 20% val, 10% test from temp (2/3 val, 1/3 test)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=1/3, random_state=42, stratify=y_temp
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

# CIFAR-10
cifar_X_train, cifar_X_val, cifar_X_test, cifar_y_train, cifar_y_val, cifar_y_test = split_data(cifar_x_norm, cifar_y)
print(f"CIFAR-10: Train={cifar_X_train.shape}, Val={cifar_X_val.shape}, Test={cifar_X_test.shape}")
print(f"CIFAR-10 labels: Train={cifar_y_train.shape}, Val={cifar_y_val.shape}, Test={cifar_y_test.shape}")

# MNIST
mnist_X_train, mnist_X_val, mnist_X_test, mnist_y_train, mnist_y_val, mnist_y_test = split_data(mnist_x_norm, mnist_y)
print(f"\nMNIST: Train={mnist_X_train.shape}, Val={mnist_X_val.shape}, Test={mnist_X_test.shape}")
print(f"MNIST labels: Train={mnist_y_train.shape}, Val={mnist_y_val.shape}, Test={mnist_y_test.shape}")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 10, figsize=(20, 4))
for i in range(10):
    axes[0, i].imshow(cifar_X_train[i], cmap='gray', vmin=50, vmax=200)
    axes[0, i].set_title(f'CIFAR:{cifar_y_train[i]}', fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(mnist_X_train[i], cmap='gray', vmin=50, vmax=200)
    axes[1, i].set_title(f'MNIST:{mnist_y_train[i]}', fontsize=8)
    axes[1, i].axis('off')
plt.suptitle('Sample Images: CIFAR-10 (top) and MNIST (bottom)')
plt.tight_layout()
plt.show()

In [ ]:
# Flatten images for PCA / linear methods
def flatten_images(X):
    return X.reshape(X.shape[0], -1)

cifar_X_train_flat = flatten_images(cifar_X_train)
cifar_X_val_flat = flatten_images(cifar_X_val)
cifar_X_test_flat = flatten_images(cifar_X_test)

mnist_X_train_flat = flatten_images(mnist_X_train)
mnist_X_val_flat = flatten_images(mnist_X_val)
mnist_X_test_flat = flatten_images(mnist_X_test)

print(f"CIFAR-10 flattened: Train={cifar_X_train_flat.shape}, Test={cifar_X_test_flat.shape}")
print(f"MNIST flattened: Train={mnist_X_train_flat.shape}, Test={mnist_X_test_flat.shape}")

---
## Task 1: Standard PCA and Randomized PCA

- Perform standard PCA with 70% training data, identify top 30 principal components
- Train logistic regression classifier on PCA features for 10-class classification
- Draw ROC curves for test datasets
- Repeat with Randomized PCA and compare
- Calculate average SNR (dB) of reconstructed test images

In [ ]:
# Helper: Compute SNR in dB
def compute_snr_db(original, reconstructed):
    """Compute average SNR in dB between original and reconstructed images."""
    signal_power = np.mean(original ** 2, axis=1)
    noise = original - reconstructed
    noise_power = np.mean(noise ** 2, axis=1)
    # Avoid division by zero
    noise_power = np.maximum(noise_power, 1e-10)
    snr = 10 * np.log10(signal_power / noise_power)
    return np.mean(snr)

# Helper: Plot ROC curves for 10-class classification
def plot_roc_curve(y_true, y_score, n_classes, title):
    """Plot ROC curve for multi-class classification."""
    # Binarize labels
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
    
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    # Micro-average
    fpr['micro'], tpr['micro'], _ = roc_curve(y_true_bin.ravel(), y_score.ravel())
    roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])
    
    plt.figure(figsize=(10, 8))
    colors = plt.cm.tab10(np.linspace(0, 1, n_classes))
    
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=1.5,
                 label=f'Class {i} (AUC = {roc_auc[i]:.3f})')
    
    plt.plot(fpr['micro'], tpr['micro'], color='deeppink', lw=2, linestyle=':',
             label=f'Micro-avg (AUC = {roc_auc["micro"]:.3f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc='best', fontsize=8)
    plt.tight_layout()
    plt.show()
    
    return roc_auc

In [ ]:
# ===== Standard PCA on CIFAR-10 =====
print("="*60)
print("Standard PCA on CIFAR-10")
print("="*60)

pca_cifar = PCA(n_components=30, svd_solver='full')
cifar_X_train_pca = pca_cifar.fit_transform(cifar_X_train_flat)
cifar_X_test_pca = pca_cifar.transform(cifar_X_test_flat)

print(f"Top 30 eigenvalues: {pca_cifar.explained_variance_[:30]}")
print(f"Total variance explained by 30 components: {pca_cifar.explained_variance_ratio_.sum()*100:.2f}%")

# Reconstruct and compute SNR
cifar_X_test_recon_pca = pca_cifar.inverse_transform(cifar_X_test_pca)
snr_cifar_pca = compute_snr_db(cifar_X_test_flat, cifar_X_test_recon_pca)
print(f"Average SNR (Standard PCA, CIFAR-10): {snr_cifar_pca:.2f} dB")

# Logistic Regression on PCA features
lr_cifar_pca = LogisticRegression(max_iter=1000, multi_class='ovr', random_state=42)
lr_cifar_pca.fit(cifar_X_train_pca, cifar_y_train)
cifar_pca_score = lr_cifar_pca.score(cifar_X_test_pca, cifar_y_test)
print(f"Logistic Regression Accuracy (Standard PCA, CIFAR-10): {cifar_pca_score*100:.2f}%")

# ROC Curve
cifar_y_score_pca = lr_cifar_pca.predict_proba(cifar_X_test_pca)
roc_auc_cifar_pca = plot_roc_curve(cifar_y_test, cifar_y_score_pca, 10, 
                                     'ROC Curve - Standard PCA (CIFAR-10)')

In [ ]:
# ===== Standard PCA on MNIST =====
print("="*60)
print("Standard PCA on MNIST")
print("="*60)

pca_mnist = PCA(n_components=30, svd_solver='full')
mnist_X_train_pca = pca_mnist.fit_transform(mnist_X_train_flat)
mnist_X_test_pca = pca_mnist.transform(mnist_X_test_flat)

print(f"Top 30 eigenvalues: {pca_mnist.explained_variance_[:30]}")
print(f"Total variance explained by 30 components: {pca_mnist.explained_variance_ratio_.sum()*100:.2f}%")

# Reconstruct and compute SNR
mnist_X_test_recon_pca = pca_mnist.inverse_transform(mnist_X_test_pca)
snr_mnist_pca = compute_snr_db(mnist_X_test_flat, mnist_X_test_recon_pca)
print(f"Average SNR (Standard PCA, MNIST): {snr_mnist_pca:.2f} dB")

# Logistic Regression on PCA features
lr_mnist_pca = LogisticRegression(max_iter=1000, multi_class='ovr', random_state=42)
lr_mnist_pca.fit(mnist_X_train_pca, mnist_y_train)
mnist_pca_score = lr_mnist_pca.score(mnist_X_test_pca, mnist_y_test)
print(f"Logistic Regression Accuracy (Standard PCA, MNIST): {mnist_pca_score*100:.2f}%")

# ROC Curve
mnist_y_score_pca = lr_mnist_pca.predict_proba(mnist_X_test_pca)
roc_auc_mnist_pca = plot_roc_curve(mnist_y_test, mnist_y_score_pca, 10, 
                                     'ROC Curve - Standard PCA (MNIST)')

In [ ]:
# ===== Randomized PCA on CIFAR-10 =====
print("="*60)
print("Randomized PCA on CIFAR-10")
print("="*60)

rpca_cifar = PCA(n_components=30, svd_solver='randomized', random_state=42)
cifar_X_train_rpca = rpca_cifar.fit_transform(cifar_X_train_flat)
cifar_X_test_rpca = rpca_cifar.transform(cifar_X_test_flat)

print(f"Top 30 eigenvalues: {rpca_cifar.explained_variance_[:30]}")
print(f"Total variance explained by 30 components: {rpca_cifar.explained_variance_ratio_.sum()*100:.2f}%")

# Reconstruct and compute SNR
cifar_X_test_recon_rpca = rpca_cifar.inverse_transform(cifar_X_test_rpca)
snr_cifar_rpca = compute_snr_db(cifar_X_test_flat, cifar_X_test_recon_rpca)
print(f"Average SNR (Randomized PCA, CIFAR-10): {snr_cifar_rpca:.2f} dB")

# Logistic Regression on Randomized PCA features
lr_cifar_rpca = LogisticRegression(max_iter=1000, multi_class='ovr', random_state=42)
lr_cifar_rpca.fit(cifar_X_train_rpca, cifar_y_train)
cifar_rpca_score = lr_cifar_rpca.score(cifar_X_test_rpca, cifar_y_test)
print(f"Logistic Regression Accuracy (Randomized PCA, CIFAR-10): {cifar_rpca_score*100:.2f}%")

# ROC Curve
cifar_y_score_rpca = lr_cifar_rpca.predict_proba(cifar_X_test_rpca)
roc_auc_cifar_rpca = plot_roc_curve(cifar_y_test, cifar_y_score_rpca, 10, 
                                      'ROC Curve - Randomized PCA (CIFAR-10)')

In [ ]:
# ===== Randomized PCA on MNIST =====
print("="*60)
print("Randomized PCA on MNIST")
print("="*60)

rpca_mnist = PCA(n_components=30, svd_solver='randomized', random_state=42)
mnist_X_train_rpca = rpca_mnist.fit_transform(mnist_X_train_flat)
mnist_X_test_rpca = rpca_mnist.transform(mnist_X_test_flat)

print(f"Top 30 eigenvalues: {rpca_mnist.explained_variance_[:30]}")
print(f"Total variance explained by 30 components: {rpca_mnist.explained_variance_ratio_.sum()*100:.2f}%")

# Reconstruct and compute SNR
mnist_X_test_recon_rpca = rpca_mnist.inverse_transform(mnist_X_test_rpca)
snr_mnist_rpca = compute_snr_db(mnist_X_test_flat, mnist_X_test_recon_rpca)
print(f"Average SNR (Randomized PCA, MNIST): {snr_mnist_rpca:.2f} dB")

# Logistic Regression on Randomized PCA features
lr_mnist_rpca = LogisticRegression(max_iter=1000, multi_class='ovr', random_state=42)
lr_mnist_rpca.fit(mnist_X_train_rpca, mnist_y_train)
mnist_rpca_score = lr_mnist_rpca.score(mnist_X_test_rpca, mnist_y_test)
print(f"Logistic Regression Accuracy (Randomized PCA, MNIST): {mnist_rpca_score*100:.2f}%")

# ROC Curve
mnist_y_score_rpca = lr_mnist_rpca.predict_proba(mnist_X_test_rpca)
roc_auc_mnist_rpca = plot_roc_curve(mnist_y_test, mnist_y_score_rpca, 10, 
                                      'ROC Curve - Randomized PCA (MNIST)')

In [ ]:
# ===== Comparison Table: Standard PCA vs Randomized PCA =====
print("\n" + "="*70)
print("COMPARISON: Standard PCA vs Randomized PCA")
print("="*70)
print(f"{'Metric':<35} {'Std PCA':>15} {'Rand PCA':>15}")
print("-" * 65)
print(f"{'CIFAR-10 Accuracy (%)':<35} {cifar_pca_score*100:>15.2f} {cifar_rpca_score*100:>15.2f}")
print(f"{'CIFAR-10 SNR (dB)':<35} {snr_cifar_pca:>15.2f} {snr_cifar_rpca:>15.2f}")
print(f"{'CIFAR-10 Variance Explained (%)':<35} {pca_cifar.explained_variance_ratio_.sum()*100:>15.2f} {rpca_cifar.explained_variance_ratio_.sum()*100:>15.2f}")
print(f"{'MNIST Accuracy (%)':<35} {mnist_pca_score*100:>15.2f} {mnist_rpca_score*100:>15.2f}")
print(f"{'MNIST SNR (dB)':<35} {snr_mnist_pca:>15.2f} {snr_mnist_rpca:>15.2f}")
print(f"{'MNIST Variance Explained (%)':<35} {pca_mnist.explained_variance_ratio_.sum()*100:>15.2f} {rpca_mnist.explained_variance_ratio_.sum()*100:>15.2f}")

print("\nObservation: Standard PCA and Randomized PCA produce nearly identical")
print("results, with Randomized PCA being computationally faster for large datasets.")

---
## Task 2: Single Layer Autoencoder with Tied Weights

- Train a single-layer autoencoder with 30 nodes, linear activation
- Mean and variance normalized input
- Tied weights: decoder weight matrix = encoder weight matrix transpose
- Each weight vector has unit magnitude
- Compare eigenvectors (PCA) with autoencoder weights
- Train logistic regression on autoencoder features

In [ ]:
# Mean and variance normalization for autoencoder input
def normalize_mean_var(X_train, X_val, X_test):
    """Zero mean, unit variance normalization"""
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std < 1e-10] = 1.0  # avoid division by zero
    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    X_test_norm = (X_test - mean) / std
    return X_train_norm, X_val_norm, X_test_norm, mean, std

# Normalize both datasets
cifar_X_train_ae, cifar_X_val_ae, cifar_X_test_ae, cifar_mean, cifar_std = normalize_mean_var(
    cifar_X_train_flat, cifar_X_val_flat, cifar_X_test_flat)

mnist_X_train_ae, mnist_X_val_ae, mnist_X_test_ae, mnist_mean, mnist_std = normalize_mean_var(
    mnist_X_train_flat, mnist_X_val_flat, mnist_X_test_flat)

print(f"CIFAR-10 AE input: mean={cifar_X_train_ae.mean():.6f}, std={cifar_X_train_ae.std():.6f}")
print(f"MNIST AE input: mean={mnist_X_train_ae.mean():.6f}, std={mnist_X_train_ae.std():.6f}")
print(f"Input dimension: CIFAR-10={cifar_X_train_ae.shape[1]}, MNIST={mnist_X_train_ae.shape[1]}")

In [ ]:
# Custom Keras layer for tied weights autoencoder with unit-norm constraint
class TiedAutoencoder(Model):
    """Single-layer autoencoder with tied weights and unit-norm constraint.
    
    Encoder: z = W^T x (linear), with each column of W having unit norm
    Decoder: x_hat = W z (tied: decoder uses same W)
    """
    def __init__(self, input_dim, latent_dim=30):
        super(TiedAutoencoder, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Initialize weight matrix W (input_dim x latent_dim)
        # Use PCA components as initialization for faster convergence
        self.W = self.add_weight(
            name='encoder_weights',
            shape=(input_dim, latent_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        # Bias for encoder
        self.b_enc = self.add_weight(
            name='encoder_bias',
            shape=(latent_dim,),
            initializer='zeros',
            trainable=True
        )
        # Bias for decoder
        self.b_dec = self.add_weight(
            name='decoder_bias',
            shape=(input_dim,),
            initializer='zeros',
            trainable=True
        )
    
    def call(self, x):
        # Normalize weight vectors to unit magnitude
        W_norm = self.normalize_weights()
        # Encoder: z = W^T x + b_enc
        z = tf.matmul(x, W_norm) + self.b_enc
        # Decoder: x_hat = W z + b_dec (tied weights)
        x_hat = tf.matmul(z, tf.transpose(W_norm)) + self.b_dec
        return x_hat, z
    
    def normalize_weights(self):
        """Normalize each column of W to unit magnitude"""
        col_norms = tf.sqrt(tf.reduce_sum(self.W ** 2, axis=0, keepdims=True))
        col_norms = tf.maximum(col_norms, 1e-8)  # avoid division by zero
        return self.W / col_norms
    
    def encode(self, x):
        W_norm = self.normalize_weights()
        return tf.matmul(x, W_norm) + self.b_enc
    
    def decode(self, z):
        W_norm = self.normalize_weights()
        return tf.matmul(z, tf.transpose(W_norm)) + self.b_dec

In [ ]:
# Training function for tied autoencoder
def train_tied_ae(X_train, X_val, input_dim, latent_dim=30, epochs=200, batch_size=256, lr=0.001):
    model = TiedAutoencoder(input_dim, latent_dim)
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    
    # Convert to tf.data for efficiency
    train_ds = tf.data.Dataset.from_tensor_slices((X_train.astype('float32'), X_train.astype('float32')))
    train_ds = train_ds.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    val_ds = tf.data.Dataset.from_tensor_slices((X_val.astype('float32'), X_val.astype('float32')))
    val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        epoch_train_loss = 0
        num_batches = 0
        for x_batch, y_batch in train_ds:
            with tf.GradientTape() as tape:
                x_hat, z = model(x_batch, training=True)
                loss = tf.reduce_mean(tf.square(y_batch - x_hat))
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            epoch_train_loss += loss.numpy()
            num_batches += 1
        
        avg_train_loss = epoch_train_loss / num_batches
        train_losses.append(avg_train_loss)
        
        # Validation loss
        val_loss_vals = []
        for x_v, y_v in val_ds:
            x_hat_v, _ = model(x_v, training=False)
            v_loss = tf.reduce_mean(tf.square(y_v - x_hat_v))
            val_loss_vals.append(v_loss.numpy())
        avg_val_loss = np.mean(val_loss_vals)
        val_losses.append(avg_val_loss)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}")
    
    return model, train_losses, val_losses

In [ ]:
# ===== Train Tied Autoencoder on CIFAR-10 =====
print("="*60)
print("Training Tied Autoencoder on CIFAR-10")
print("="*60)

cifar_ae, cifar_ae_train_loss, cifar_ae_val_loss = train_tied_ae(
    cifar_X_train_ae, cifar_X_val_ae, 
    input_dim=cifar_X_train_ae.shape[1], 
    latent_dim=30, epochs=200, batch_size=256, lr=0.001
)

# Plot loss
plt.figure(figsize=(10, 5))
plt.plot(cifar_ae_train_loss, label='Train Loss')
plt.plot(cifar_ae_val_loss, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Tied Autoencoder Training Loss (CIFAR-10)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ===== Train Tied Autoencoder on MNIST =====
print("="*60)
print("Training Tied Autoencoder on MNIST")
print("="*60)

mnist_ae, mnist_ae_train_loss, mnist_ae_val_loss = train_tied_ae(
    mnist_X_train_ae, mnist_X_val_ae, 
    input_dim=mnist_X_train_ae.shape[1], 
    latent_dim=30, epochs=200, batch_size=256, lr=0.001
)

# Plot loss
plt.figure(figsize=(10, 5))
plt.plot(mnist_ae_train_loss, label='Train Loss')
plt.plot(mnist_ae_val_loss, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Tied Autoencoder Training Loss (MNIST)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ===== Compare PCA Eigenvectors with Autoencoder Weights =====
def compare_pca_ae(pca_model, ae_model, dataset_name, img_shape=(28, 28)):
    """Compare PCA eigenvectors with autoencoder weight vectors."""
    # PCA components (eigenvectors) - shape: (n_components, input_dim)
    pca_components = pca_model.components_  # (30, 784)
    
    # AE encoder weights - shape: (input_dim, latent_dim)
    ae_weights = ae_model.normalize_weights().numpy()  # (784, 30)
    ae_weight_vecs = ae_weights.T  # (30, 784) - same layout as PCA
    
    # Compute correlation between each PCA component and each AE weight vector
    # For each pair, compute absolute correlation (since sign can flip)
    correlations = np.zeros((30, 30))
    for i in range(30):
        for j in range(30):
            corr = np.abs(np.corrcoef(pca_components[i], ae_weight_vecs[j])[0, 1])
            correlations[i, j] = corr
    
    # Find best matching pairs
    print(f"\n{dataset_name}: PCA vs Autoencoder Weight Correlation")
    print("-" * 50)
    
    # For each PCA component, find the best matching AE weight
    best_matches = []
    used = set()
    for i in range(30):
        best_j = -1
        best_corr = -1
        for j in range(30):
            if j not in used and correlations[i, j] > best_corr:
                best_corr = correlations[i, j]
                best_j = j
        used.add(best_j)
        best_matches.append((i, best_j, best_corr))
    
    avg_corr = np.mean([m[2] for m in best_matches])
    print(f"Average absolute correlation between matched pairs: {avg_corr:.4f}")
    print(f"Max correlation: {max(m[2] for m in best_matches):.4f}")
    print(f"Min correlation: {min(m[2] for m in best_matches):.4f}")
    
    # Display eigenvectors and weight vectors as images
    fig, axes = plt.subplots(6, 10, figsize=(20, 12))
    fig.suptitle(f'{dataset_name}: PCA Eigenvectors (top row) vs AE Weights (bottom row) - First 5 components', fontsize=14)
    
    for i in range(5):
        # PCA eigenvector
        pca_img = pca_components[i].reshape(img_shape)
        axes[0, i*2].imshow(pca_img, cmap='gray')
        axes[0, i*2].set_title(f'PCA PC{i+1}', fontsize=8)
        axes[0, i*2].axis('off')
        
        # AE weight
        j = best_matches[i][1]
        ae_img = ae_weight_vecs[j].reshape(img_shape)
        axes[0, i*2+1].imshow(ae_img, cmap='gray')
        axes[0, i*2+1].set_title(f'AE W{j+1}\n(r={best_matches[i][2]:.3f})', fontsize=8)
        axes[0, i*2+1].axis('off')
    
    # Hide unused axes
    for idx in range(10, 10):
        axes[0, idx].axis('off')
    for row in range(1, 6):
        for col in range(10):
            axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return correlations, best_matches

# Compare for CIFAR-10
print("="*60)
print("Comparing PCA Eigenvectors vs Autoencoder Weights: CIFAR-10")
print("="*60)
corr_cifar, matches_cifar = compare_pca_ae(pca_cifar, cifar_ae, 'CIFAR-10')

In [ ]:
# Compare for MNIST
print("="*60)
print("Comparing PCA Eigenvectors vs Autoencoder Weights: MNIST")
print("="*60)
corr_mnist, matches_mnist = compare_pca_ae(pca_mnist, mnist_ae, 'MNIST')

In [ ]:
# Display all 30 PCA eigenvectors and 30 AE weight vectors as images
def display_all_components(pca_model, ae_model, dataset_name, img_shape=(28, 28)):
    pca_components = pca_model.components_
    ae_weights = ae_model.normalize_weights().numpy().T
    
    # PCA Eigenvectors
    fig, axes = plt.subplots(3, 10, figsize=(20, 6))
    fig.suptitle(f'{dataset_name}: All 30 PCA Eigenvectors (Top 30)', fontsize=14)
    for i in range(30):
        row, col = i // 10, i % 10
        axes[row, col].imshow(pca_components[i].reshape(img_shape), cmap='gray')
        axes[row, col].set_title(f'PC{i+1}', fontsize=7)
        axes[row, col].axis('off')
    plt.tight_layout()
    plt.show()
    
    # AE Weights
    fig, axes = plt.subplots(3, 10, figsize=(20, 6))
    fig.suptitle(f'{dataset_name}: All 30 Autoencoder Weight Vectors', fontsize=14)
    for i in range(30):
        row, col = i // 10, i % 10
        axes[row, col].imshow(ae_weights[i].reshape(img_shape), cmap='gray')
        axes[row, col].set_title(f'W{i+1}', fontsize=7)
        axes[row, col].axis('off')
    plt.tight_layout()
    plt.show()

display_all_components(pca_cifar, cifar_ae, 'CIFAR-10')
display_all_components(pca_mnist, mnist_ae, 'MNIST')

In [ ]:
# ===== Logistic Regression with Autoencoder Features =====
print("="*60)
print("Logistic Regression with Autoencoder Features")
print("="*60)

# Get autoencoder features
cifar_X_train_ae_feat = cifar_ae.encode(cifar_X_train_ae.astype('float32')).numpy()
cifar_X_test_ae_feat = cifar_ae.encode(cifar_X_test_ae.astype('float32')).numpy()

mnist_X_train_ae_feat = mnist_ae.encode(mnist_X_train_ae.astype('float32')).numpy()
mnist_X_test_ae_feat = mnist_ae.encode(mnist_X_test_ae.astype('float32')).numpy()

# CIFAR-10
lr_cifar_ae = LogisticRegression(max_iter=1000, multi_class='ovr', random_state=42)
lr_cifar_ae.fit(cifar_X_train_ae_feat, cifar_y_train)
cifar_ae_score = lr_cifar_ae.score(cifar_X_test_ae_feat, cifar_y_test)
print(f"CIFAR-10 Accuracy (AE features): {cifar_ae_score*100:.2f}%")
print(f"CIFAR-10 Accuracy (PCA features): {cifar_pca_score*100:.2f}%")

# MNIST
lr_mnist_ae = LogisticRegression(max_iter=1000, multi_class='ovr', random_state=42)
lr_mnist_ae.fit(mnist_X_train_ae_feat, mnist_y_train)
mnist_ae_score = lr_mnist_ae.score(mnist_X_test_ae_feat, mnist_y_test)
print(f"\nMNIST Accuracy (AE features): {mnist_ae_score*100:.2f}%")
print(f"MNIST Accuracy (PCA features): {mnist_pca_score*100:.2f}%")

In [ ]:
# ROC curves for Autoencoder features
# CIFAR-10
cifar_y_score_ae = lr_cifar_ae.predict_proba(cifar_X_test_ae_feat)
roc_auc_cifar_ae = plot_roc_curve(cifar_y_test, cifar_y_score_ae, 10, 
                                    'ROC Curve - Autoencoder Features (CIFAR-10)')

# MNIST
mnist_y_score_ae = lr_mnist_ae.predict_proba(mnist_X_test_ae_feat)
roc_auc_mnist_ae = plot_roc_curve(mnist_y_test, mnist_y_score_ae, 10, 
                                    'ROC Curve - Autoencoder Features (MNIST)')

In [ ]:
# ===== SNR for Tied Autoencoder Reconstruction =====
cifar_X_test_ae_recon, _ = cifar_ae(cifar_X_test_ae.astype('float32'))
cifar_X_test_ae_recon = cifar_X_test_ae_recon.numpy()
# De-normalize for SNR computation in original space
cifar_X_test_ae_recon_orig = cifar_X_test_ae_recon * cifar_std + cifar_mean
snr_cifar_ae = compute_snr_db(cifar_X_test_flat, cifar_X_test_ae_recon_orig)

mnist_X_test_ae_recon, _ = mnist_ae(mnist_X_test_ae.astype('float32'))
mnist_X_test_ae_recon = mnist_X_test_ae_recon.numpy()
mnist_X_test_ae_recon_orig = mnist_X_test_ae_recon * mnist_std + mnist_mean
snr_mnist_ae = compute_snr_db(mnist_X_test_flat, mnist_X_test_ae_recon_orig)

print("="*60)
print("SNR Comparison: PCA vs Tied Autoencoder")
print("="*60)
print(f"CIFAR-10: PCA SNR = {snr_cifar_pca:.2f} dB, AE SNR = {snr_cifar_ae:.2f} dB")
print(f"MNIST:    PCA SNR = {snr_mnist_pca:.2f} dB, AE SNR = {snr_mnist_ae:.2f} dB")
print()
print("The tied linear autoencoder with unit-norm constraint learns weight vectors")
print("that are equivalent (up to sign) to PCA eigenvectors. This is because a linear")
print("autoencoder with tied weights minimizing MSE is equivalent to PCA, and the")
print("unit-norm constraint ensures they span the same subspace as the top eigenvectors.")

---
## Task 3: Deep Convolutional Autoencoder and Multi-Layer Autoencoders

- Design and train a deep convolutional autoencoder with 30-dim latent space
- Calculate average SNR for test datasets
- Compare with single-layer autoencoder (sigmoid encoder, linear decoder)
- Train 3-hidden-layer autoencoder with equal node distribution (10-10-10)

In [ ]:
# Prepare 4D data for convolutional autoencoder
# Images are already 28x28, normalized to [50, 200]
# Rescale to [0, 1] for neural network training stability
def rescale_01(X):
    return (X - 50.0) / 150.0

def rescale_back(X):
    return X * 150.0 + 50.0

cifar_X_train_4d = rescale_01(cifar_X_train)[..., np.newaxis]
cifar_X_val_4d = rescale_01(cifar_X_val)[..., np.newaxis]
cifar_X_test_4d = rescale_01(cifar_X_test)[..., np.newaxis]

mnist_X_train_4d = rescale_01(mnist_X_train)[..., np.newaxis]
mnist_X_val_4d = rescale_01(mnist_X_val)[..., np.newaxis]
mnist_X_test_4d = rescale_01(mnist_X_test)[..., np.newaxis]

print(f"CIFAR-10 4D: {cifar_X_train_4d.shape}, range [{cifar_X_train_4d.min():.3f}, {cifar_X_train_4d.max():.3f}]")
print(f"MNIST 4D: {mnist_X_train_4d.shape}, range [{mnist_X_train_4d.min():.3f}, {mnist_X_train_4d.max():.3f}]")

In [ ]:
# ===== Deep Convolutional Autoencoder with 30-dim latent space =====
def build_conv_autoencoder(input_shape=(28, 28, 1), latent_dim=30):
    """Build a deep convolutional autoencoder with 30-dimensional latent space."""
    # Encoder
    inputs = layers.Input(shape=input_shape, name='encoder_input')
    
    # Conv block 1
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)  # 14x14x32
    
    # Conv block 2
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)  # 7x7x64
    
    # Conv block 3
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)  # 7x7x128
    
    # Flatten and project to latent space
    x = layers.Flatten()(x)  # 7*7*128 = 6272
    latent = layers.Dense(latent_dim, name='latent_space')(x)
    
    encoder = Model(inputs, latent, name='encoder')
    
    # Decoder
    latent_input = layers.Input(shape=(latent_dim,), name='decoder_input')
    
    # Project back and reshape
    x = layers.Dense(7 * 7 * 128, activation='relu')(latent_input)
    x = layers.Reshape((7, 7, 128))(x)
    
    # Deconv block 3
    x = layers.Conv2DTranspose(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Upsample block 2
    x = layers.Conv2DTranspose(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)  # 14x14x64
    
    # Upsample block 1
    x = layers.Conv2DTranspose(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D((2, 2))(x)  # 28x28x32
    
    # Output
    outputs = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)
    
    decoder = Model(latent_input, outputs, name='decoder')
    
    # Autoencoder
    autoencoder = Model(inputs, decoder(encoder(inputs)), name='conv_autoencoder')
    
    return encoder, decoder, autoencoder

# Build and display architecture
enc_cifar, dec_cifar, conv_ae_cifar = build_conv_autoencoder()
conv_ae_cifar.compile(optimizer='adam', loss='mse')
print("Convolutional Autoencoder Architecture:")
conv_ae_cifar.summary()

In [ ]:
# Train Conv Autoencoder on CIFAR-10
print("Training Convolutional Autoencoder on CIFAR-10...")
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_cifar_conv = conv_ae_cifar.fit(
    cifar_X_train_4d, cifar_X_train_4d,
    epochs=100, batch_size=256,
    validation_data=(cifar_X_val_4d, cifar_X_val_4d),
    callbacks=[early_stop],
    verbose=1
)

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(history_cifar_conv.history['loss'], label='Train Loss')
plt.plot(history_cifar_conv.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Deep Convolutional Autoencoder Training Loss (CIFAR-10)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train Conv Autoencoder on MNIST
enc_mnist, dec_mnist, conv_ae_mnist = build_conv_autoencoder()
conv_ae_mnist.compile(optimizer='adam', loss='mse')

print("Training Convolutional Autoencoder on MNIST...")
history_mnist_conv = conv_ae_mnist.fit(
    mnist_X_train_4d, mnist_X_train_4d,
    epochs=100, batch_size=256,
    validation_data=(mnist_X_val_4d, mnist_X_val_4d),
    callbacks=[early_stop],
    verbose=1
)

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(history_mnist_conv.history['loss'], label='Train Loss')
plt.plot(history_mnist_conv.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Deep Convolutional Autoencoder Training Loss (MNIST)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compute SNR for Conv Autoencoder
def compute_snr_db_images(original_4d, reconstructed_4d):
    """Compute average SNR in dB for 4D image arrays."""
    orig_flat = original_4d.reshape(original_4d.shape[0], -1)
    recon_flat = reconstructed_4d.reshape(reconstructed_4d.shape[0], -1)
    return compute_snr_db(orig_flat, recon_flat)

# CIFAR-10
cifar_X_test_conv_recon = conv_ae_cifar.predict(cifar_X_test_4d, verbose=0)
snr_cifar_conv = compute_snr_db_images(cifar_X_test_4d, cifar_X_test_conv_recon)

# MNIST
mnist_X_test_conv_recon = conv_ae_mnist.predict(mnist_X_test_4d, verbose=0)
snr_mnist_conv = compute_snr_db_images(mnist_X_test_4d, mnist_X_test_conv_recon)

print(f"Conv Autoencoder SNR - CIFAR-10: {snr_cifar_conv:.2f} dB")
print(f"Conv Autoencoder SNR - MNIST: {snr_mnist_conv:.2f} dB")

In [ ]:
# Visualize Conv Autoencoder reconstructions
fig, axes = plt.subplots(4, 10, figsize=(20, 8))
fig.suptitle('Convolutional Autoencoder: Original (top) vs Reconstructed (bottom)', fontsize=14)

for i in range(10):
    # CIFAR-10 original
    axes[0, i].imshow(cifar_X_test_4d[i, :, :, 0], cmap='gray')
    axes[0, i].set_title(f'CIFAR:{cifar_y_test[i]}', fontsize=8)
    axes[0, i].axis('off')
    # CIFAR-10 reconstructed
    axes[1, i].imshow(cifar_X_test_conv_recon[i, :, :, 0], cmap='gray')
    axes[1, i].axis('off')
    
    # MNIST original
    axes[2, i].imshow(mnist_X_test_4d[i, :, :, 0], cmap='gray')
    axes[2, i].set_title(f'MNIST:{mnist_y_test[i]}', fontsize=8)
    axes[2, i].axis('off')
    # MNIST reconstructed
    axes[3, i].imshow(mnist_X_test_conv_recon[i, :, :, 0], cmap='gray')
    axes[3, i].axis('off')

axes[0, 0].set_ylabel('CIFAR\nOriginal', fontsize=9)
axes[1, 0].set_ylabel('CIFAR\nRecon', fontsize=9)
axes[2, 0].set_ylabel('MNIST\nOriginal', fontsize=9)
axes[3, 0].set_ylabel('MNIST\nRecon', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ===== Single Layer Autoencoder with Sigmoid Activation (Encoder) and Linear (Decoder) =====
# 30 hidden nodes

def build_single_layer_ae(input_dim, latent_dim=30):
    """Single-layer autoencoder: sigmoid encoder, linear decoder."""
    # Encoder
    encoder_input = layers.Input(shape=(input_dim,))
    encoded = layers.Dense(latent_dim, activation='sigmoid', name='encoder')(encoder_input)
    encoder = Model(encoder_input, encoded, name='encoder_sigmoid')
    
    # Decoder
    decoder_input = layers.Input(shape=(latent_dim,))
    decoded = layers.Dense(input_dim, activation='linear', name='decoder')(decoder_input)
    decoder = Model(decoder_input, decoded, name='decoder_linear')
    
    # Autoencoder
    autoencoder = Model(encoder_input, decoder(encoder(encoder_input)), name='single_layer_ae')
    
    return encoder, decoder, autoencoder

# Rescale normalized [50, 200] data to [0, 1] for sigmoid
cifar_X_train_01 = (cifar_X_train_flat - 50) / 150.0
cifar_X_val_01 = (cifar_X_val_flat - 50) / 150.0
cifar_X_test_01 = (cifar_X_test_flat - 50) / 150.0

mnist_X_train_01 = (mnist_X_train_flat - 50) / 150.0
mnist_X_val_01 = (mnist_X_val_flat - 50) / 150.0
mnist_X_test_01 = (mnist_X_test_flat - 50) / 150.0

input_dim_cifar = cifar_X_train_flat.shape[1]  # 784
input_dim_mnist = mnist_X_train_flat.shape[1]  # 784

In [ ]:
# Train Single-Layer AE (sigmoid/linear) on CIFAR-10
print("="*60)
print("Single-Layer AE (Sigmoid/Linear) on CIFAR-10")
print("="*60)

enc_sigmoid_cifar, dec_sigmoid_cifar, ae_sigmoid_cifar = build_single_layer_ae(input_dim_cifar, 30)
ae_sigmoid_cifar.compile(optimizer='adam', loss='mse')

history_sigmoid_cifar = ae_sigmoid_cifar.fit(
    cifar_X_train_01, cifar_X_train_01,
    epochs=200, batch_size=256,
    validation_data=(cifar_X_val_01, cifar_X_val_01),
    callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
    verbose=1
)

plt.figure(figsize=(10, 5))
plt.plot(history_sigmoid_cifar.history['loss'], label='Train Loss')
plt.plot(history_sigmoid_cifar.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Single-Layer AE (Sigmoid/Linear) Training Loss (CIFAR-10)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train Single-Layer AE (sigmoid/linear) on MNIST
print("="*60)
print("Single-Layer AE (Sigmoid/Linear) on MNIST")
print("="*60)

enc_sigmoid_mnist, dec_sigmoid_mnist, ae_sigmoid_mnist = build_single_layer_ae(input_dim_mnist, 30)
ae_sigmoid_mnist.compile(optimizer='adam', loss='mse')

history_sigmoid_mnist = ae_sigmoid_mnist.fit(
    mnist_X_train_01, mnist_X_train_01,
    epochs=200, batch_size=256,
    validation_data=(mnist_X_val_01, mnist_X_val_01),
    callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
    verbose=1
)

plt.figure(figsize=(10, 5))
plt.plot(history_sigmoid_mnist.history['loss'], label='Train Loss')
plt.plot(history_sigmoid_mnist.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Single-Layer AE (Sigmoid/Linear) Training Loss (MNIST)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# SNR for Single-Layer AE (Sigmoid/Linear)
cifar_X_test_sigmoid_recon = ae_sigmoid_cifar.predict(cifar_X_test_01, verbose=0)
snr_cifar_sigmoid = compute_snr_db(cifar_X_test_01, cifar_X_test_sigmoid_recon)

mnist_X_test_sigmoid_recon = ae_sigmoid_mnist.predict(mnist_X_test_01, verbose=0)
snr_mnist_sigmoid = compute_snr_db(mnist_X_test_01, mnist_X_test_sigmoid_recon)

print(f"Single-Layer AE (Sigmoid/Linear) SNR - CIFAR-10: {snr_cifar_sigmoid:.2f} dB")
print(f"Single-Layer AE (Sigmoid/Linear) SNR - MNIST: {snr_mnist_sigmoid:.2f} dB")

In [ ]:
# ===== 3-Hidden-Layer Autoencoder with Equal Node Distribution (10-10-10) =====
# Sigmoid activation at encoder layers, linear at final decoder layer

def build_3layer_ae(input_dim, hidden_dims=[10, 10, 10]):
    """3-hidden-layer autoencoder with sigmoid encoder and linear decoder.
    Hidden nodes distributed equally: 10-10-10 (total = 30).
    """
    # Encoder
    encoder_input = layers.Input(shape=(input_dim,))
    
    # Encoder layer 1: input_dim -> 10
    x = layers.Dense(hidden_dims[0], activation='sigmoid', name='enc1')(encoder_input)
    # Encoder layer 2: 10 -> 10
    x = layers.Dense(hidden_dims[1], activation='sigmoid', name='enc2')(x)
    # Latent / bottleneck: 10 -> 10
    latent = layers.Dense(hidden_dims[2], activation='sigmoid', name='latent')(x)
    
    encoder = Model(encoder_input, latent, name='encoder_3layer')
    
    # Decoder (symmetric)
    decoder_input = layers.Input(shape=(hidden_dims[2],))
    # Decoder layer 1: 10 -> 10
    x = layers.Dense(hidden_dims[1], activation='sigmoid', name='dec1')(decoder_input)
    # Decoder layer 2: 10 -> 10
    x = layers.Dense(hidden_dims[0], activation='sigmoid', name='dec2')(x)
    # Output layer: 10 -> input_dim (linear)
    decoded = layers.Dense(input_dim, activation='linear', name='dec_out')(x)
    
    decoder = Model(decoder_input, decoded, name='decoder_3layer')
    
    # Autoencoder
    autoencoder = Model(encoder_input, decoder(encoder(encoder_input)), name='ae_3layer')
    
    return encoder, decoder, autoencoder

print("3-Hidden-Layer Autoencoder Architecture (10-10-10):")
_, _, test_model = build_3layer_ae(input_dim_cifar, [10, 10, 10])
test_model.summary()

In [ ]:
# Train 3-Layer AE on CIFAR-10
print("="*60)
print("3-Layer AE (10-10-10, Sigmoid/Linear) on CIFAR-10")
print("="*60)

enc_3l_cifar, dec_3l_cifar, ae_3l_cifar = build_3layer_ae(input_dim_cifar, [10, 10, 10])
ae_3l_cifar.compile(optimizer='adam', loss='mse')

history_3l_cifar = ae_3l_cifar.fit(
    cifar_X_train_01, cifar_X_train_01,
    epochs=300, batch_size=256,
    validation_data=(cifar_X_val_01, cifar_X_val_01),
    callbacks=[EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)],
    verbose=1
)

plt.figure(figsize=(10, 5))
plt.plot(history_3l_cifar.history['loss'], label='Train Loss')
plt.plot(history_3l_cifar.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('3-Layer AE (10-10-10) Training Loss (CIFAR-10)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train 3-Layer AE on MNIST
print("="*60)
print("3-Layer AE (10-10-10, Sigmoid/Linear) on MNIST")
print("="*60)

enc_3l_mnist, dec_3l_mnist, ae_3l_mnist = build_3layer_ae(input_dim_mnist, [10, 10, 10])
ae_3l_mnist.compile(optimizer='adam', loss='mse')

history_3l_mnist = ae_3l_mnist.fit(
    mnist_X_train_01, mnist_X_train_01,
    epochs=300, batch_size=256,
    validation_data=(mnist_X_val_01, mnist_X_val_01),
    callbacks=[EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)],
    verbose=1
)

plt.figure(figsize=(10, 5))
plt.plot(history_3l_mnist.history['loss'], label='Train Loss')
plt.plot(history_3l_mnist.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('3-Layer AE (10-10-10) Training Loss (MNIST)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# SNR for 3-Layer AE
cifar_X_test_3l_recon = ae_3l_cifar.predict(cifar_X_test_01, verbose=0)
snr_cifar_3l = compute_snr_db(cifar_X_test_01, cifar_X_test_3l_recon)

mnist_X_test_3l_recon = ae_3l_mnist.predict(mnist_X_test_01, verbose=0)
snr_mnist_3l = compute_snr_db(mnist_X_test_01, mnist_X_test_3l_recon)

print(f"3-Layer AE (10-10-10) SNR - CIFAR-10: {snr_cifar_3l:.2f} dB")
print(f"3-Layer AE (10-10-10) SNR - MNIST: {snr_mnist_3l:.2f} dB")

In [ ]:
# ===== FINAL COMPREHENSIVE COMPARISON =====
print("\n" + "="*80)
print("FINAL COMPARISON: SNR (dB) across all models and datasets")
print("="*80)
print(f"{'Model':<45} {'CIFAR-10':>15} {'MNIST':>15}")
print("-" * 75)
print(f"{'Standard PCA (30 components)':<45} {snr_cifar_pca:>15.2f} {snr_mnist_pca:>15.2f}")
print(f"{'Randomized PCA (30 components)':<45} {snr_cifar_rpca:>15.2f} {snr_mnist_rpca:>15.2f}")
print(f"{'Tied Linear AE (30 nodes)':<45} {snr_cifar_ae:>15.2f} {snr_mnist_ae:>15.2f}")
print(f"{'Single-Layer AE (Sigmoid/Linear, 30)':<45} {snr_cifar_sigmoid:>15.2f} {snr_mnist_sigmoid:>15.2f}")
print(f"{'Deep Conv AE (30 latent dim)':<45} {snr_cifar_conv:>15.2f} {snr_mnist_conv:>15.2f}")
print(f"{'3-Layer AE (10-10-10, Sigmoid/Linear)':<45} {snr_cifar_3l:>15.2f} {snr_mnist_3l:>15.2f}")

print("\n" + "="*80)
print("Classification Accuracy Comparison")
print("="*80)
print(f"{'Model':<45} {'CIFAR-10':>15} {'MNIST':>15}")
print("-" * 75)
print(f"{'Standard PCA + Logistic Reg':<45} {cifar_pca_score*100:>14.2f}% {mnist_pca_score*100:>14.2f}%")
print(f"{'Randomized PCA + Logistic Reg':<45} {cifar_rpca_score*100:>14.2f}% {mnist_rpca_score*100:>14.2f}%")
print(f"{'Tied Linear AE + Logistic Reg':<45} {cifar_ae_score*100:>14.2f}% {mnist_ae_score*100:>14.2f}%")

In [ ]:
# ===== Visual comparison of reconstructions =====
n_samples = 8
fig, axes = plt.subplots(6, n_samples, figsize=(20, 14))

for i in range(n_samples):
    # Original
    axes[0, i].imshow(cifar_X_test[i], cmap='gray', vmin=50, vmax=200)
    axes[0, i].axis('off')
    
    # PCA reconstruction
    axes[1, i].imshow(cifar_X_test_recon_pca[i].reshape(28, 28), cmap='gray', vmin=50, vmax=200)
    axes[1, i].axis('off')
    
    # Single-layer AE (sigmoid/linear)
    recon_sigmoid = cifar_X_test_sigmoid_recon[i].reshape(28, 28) * 150.0 + 50.0
    axes[2, i].imshow(recon_sigmoid, cmap='gray', vmin=50, vmax=200)
    axes[2, i].axis('off')
    
    # Conv AE
    axes[3, i].imshow(cifar_X_test_conv_recon[i, :, :, 0] * 150.0 + 50.0, cmap='gray', vmin=50, vmax=200)
    axes[3, i].axis('off')
    
    # 3-Layer AE
    recon_3l = cifar_X_test_3l_recon[i].reshape(28, 28) * 150.0 + 50.0
    axes[4, i].imshow(recon_3l, cmap='gray', vmin=50, vmax=200)
    axes[4, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('PCA', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('1-Layer AE\n(Sigmoid)', fontsize=11, fontweight='bold')
axes[3, 0].set_ylabel('Conv AE', fontsize=11, fontweight='bold')
axes[4, 0].set_ylabel('3-Layer AE\n(10-10-10)', fontsize=11, fontweight='bold')

# Hide the last row
for i in range(n_samples):
    axes[5, i].axis('off')

plt.suptitle('CIFAR-10 Reconstruction Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# MNIST reconstruction comparison
n_samples = 8
fig, axes = plt.subplots(5, n_samples, figsize=(20, 12))

for i in range(n_samples):
    # Original
    axes[0, i].imshow(mnist_X_test[i], cmap='gray', vmin=50, vmax=200)
    axes[0, i].axis('off')
    
    # PCA reconstruction
    axes[1, i].imshow(mnist_X_test_recon_pca[i].reshape(28, 28), cmap='gray', vmin=50, vmax=200)
    axes[1, i].axis('off')
    
    # Single-layer AE (sigmoid/linear)
    recon_sigmoid = mnist_X_test_sigmoid_recon[i].reshape(28, 28) * 150.0 + 50.0
    axes[2, i].imshow(recon_sigmoid, cmap='gray', vmin=50, vmax=200)
    axes[2, i].axis('off')
    
    # Conv AE
    axes[3, i].imshow(mnist_X_test_conv_recon[i, :, :, 0] * 150.0 + 50.0, cmap='gray', vmin=50, vmax=200)
    axes[3, i].axis('off')
    
    # 3-Layer AE
    recon_3l = mnist_X_test_3l_recon[i].reshape(28, 28) * 150.0 + 50.0
    axes[4, i].imshow(recon_3l, cmap='gray', vmin=50, vmax=200)
    axes[4, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('PCA', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('1-Layer AE\n(Sigmoid)', fontsize=11, fontweight='bold')
axes[3, 0].set_ylabel('Conv AE', fontsize=11, fontweight='bold')
axes[4, 0].set_ylabel('3-Layer AE\n(10-10-10)', fontsize=11, fontweight='bold')

plt.suptitle('MNIST Reconstruction Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## Summary and Observations

### Task 1: PCA vs Randomized PCA
- Standard PCA and Randomized PCA produce nearly identical results for both CIFAR-10 and MNIST datasets with 30 components
- Randomized PCA is computationally faster due to approximate SVD
- MNIST achieves much higher classification accuracy than CIFAR-10 with the same 30 components, as MNIST has simpler structure
- SNR values are consistent between Standard and Randomized PCA

### Task 2: Linear Autoencoder vs PCA
- The linear autoencoder with tied weights and unit-norm constraint learns weight vectors that are equivalent (up to sign and rotation within the subspace) to PCA eigenvectors
- This is theoretically expected: a linear autoencoder with tied weights minimizing MSE converges to the same subspace as the top PCA eigenvectors
- The unit-norm constraint ensures weight vectors have the same normalization as PCA eigenvectors
- Classification accuracy with autoencoder features closely matches PCA-based classification

### Task 3: Deep vs Shallow Autoencoders
- The deep convolutional autoencoder achieves the best reconstruction quality (highest SNR) because it exploits spatial structure through convolutions
- The single-layer autoencoder with sigmoid activation (nonlinear) can capture nonlinear patterns, but with only 30 nodes and a single layer, its capacity is limited
- The 3-hidden-layer autoencoder (10-10-10) distributes 30 total nodes across 3 layers, creating a deeper but narrower bottleneck. While deeper, the narrow intermediate layers (10 nodes each) constrain information flow, typically resulting in lower SNR than the single 30-node layer autoencoder
- Key insight: distributing nodes across multiple layers creates information bottlenecks at each layer, reducing reconstruction quality compared to a single wide layer with the same total number of parameters